# PINN Heston


In [ ]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
from scipy.stats import norm

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim


### Colab Setup


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")


### Paths


In [ ]:
data_path = Path("..") / "data" / "generated" / "hs_collocation.parquet"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/final") # or eda depending on your structure
else:
    out_dir = Path("..") / "plots" / "final"

out_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {out_dir}")

out_path_pinn = out_dir / "pinn_hs.pdf"
out_path_loss_total = out_dir / "pinn_hs_loss_total.pdf"
out_path_loss_pde = out_dir / "pinn_hs_loss_pde.pdf"
out_path_loss_ic = out_dir / "pinn_hs_loss_ic.pdf"
out_path_loss_bc = out_dir / "pinn_hs_loss_bc.pdf"


### Global parameters


In [ ]:
# Answer to the universe and everything
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Adding torch backends
torch.backends.cudnn.benchmark = True

# Option Physics for Heston (Analytical Benchmarks)
S_max = 300.0
T_max = 1.0
K = 100.0
r = 0.05
v_min = 0.01
v_max = 1.0
v0 = 0.04 

# Heston specific parameters
kappa = 2.0    # mean-reversion speed
theta = 0.04   # long-run variance
xi = 0.3       # vol-of-vol
rho = -0.7     # correlation

# NN (best from hyperparameter sweeps)
HIDDEN_LAYERS = 3
NEURONS_PER_LAYER = 256
LEARNING_RATE = 5e-3
EPOCHS = 50000
PRINT_ROWS = 20

# PINN Loss Weights (best from sweep 3: lambda)
LAMBDA_PDE = 10.0
LAMBDA_IC = 10.0
LAMBDA_BC = 5.0


In [ ]:
import sys
sys.path.insert(0, "../scripts")
from hspinn import HSPINN


### Reading data


In [ ]:
# Loading Parquet data file into a Pandas DataFrame
df_all = pd.read_parquet(data_path)

# Helper function to extract columns and turn them into gradient-tracking Tensors
def extract_tensors(df_subset):
    S_tensor = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    v_tensor = torch.tensor(df_subset['v'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau_tensor = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S_tensor, v_tensor, tau_tensor

# Interior points
df_interior = df_all[df_all['point_type'] == 'interior']
S_in, v_in, tau_in = extract_tensors(df_interior)

# Initial Condition points (Maturity)
df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, v_ic, tau_ic = extract_tensors(df_ic)

# Boundary Condition points
df_bc = df_all[df_all['point_type'].str.startswith('boundary')]
S_bc, v_bc, tau_bc = extract_tensors(df_bc)

print(f"Read data from {data_path}")
print(f"Interior points: {len(S_in)}")
print(f"IC points:       {len(S_ic)}")
print(f"BC points:       {len(S_bc)}")


### Training a PINN


In [ ]:
model = HSPINN(hidden_layers=HIDDEN_LAYERS, neurons_per_layer=NEURONS_PER_LAYER, activation='tanh').to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


### Training Loop


In [ ]:
# Inputs
epochs = EPOCHS
print(f"Starting PINN learning at {epochs} epochs")


In [ ]:
# Initialize history lists
history = {
    'epoch': [],
    'total': [],
    'pde': [],
    'ic': [],
    'bc': []
}

# Print the table header
print(f"{'Epoch':>6} | {'Total Loss':>12} | {'PDE Loss':>12} | {'IC Loss':>12} | {'BC Loss':>12}")
print("-" * 66)

# Creating an array **once** before the loop starts to stop memory thrashing
grad_ones = torch.ones_like(S_in)

# Start that timer!
start_time = time.time()

# Looping over the epochs
for epoch in range(epochs):

    # Always zero the gradient before a new step
    optimizer.zero_grad()

    # Calculating the PDE loss
    V_pred = model(S_in, v_in, tau_in)

    # First-order derivatives
    V_S   = torch.autograd.grad(V_pred, S_in,   grad_outputs=grad_ones, create_graph=True)[0]
    V_v   = torch.autograd.grad(V_pred, v_in,   grad_outputs=grad_ones, create_graph=True)[0]
    V_tau = torch.autograd.grad(V_pred, tau_in, grad_outputs=grad_ones, create_graph=True)[0]

    # Second-order derivatives
    V_SS  = torch.autograd.grad(V_S, S_in, grad_outputs=grad_ones, create_graph=True)[0]
    V_vv  = torch.autograd.grad(V_v, v_in, grad_outputs=grad_ones, create_graph=True)[0]
    V_Sv  = torch.autograd.grad(V_S, v_in, grad_outputs=grad_ones, create_graph=True)[0]

    # Heston PDE equation
    pde_residual = V_tau - (
        0.5 * v_in * S_in**2 * V_SS
        + rho * xi * v_in * S_in * V_Sv
        + 0.5 * xi**2 * v_in * V_vv
        + r * S_in * V_S
        + kappa * (theta - v_in) * V_v
        - r * V_pred
    )
    loss_pde = torch.mean(pde_residual**2)

    # Calculating the initial condition loss (payoff at maturity)
    V_ic_pred = model(S_ic, v_ic, tau_ic)
    V_ic_true = torch.relu(S_ic - K)
    loss_ic = torch.mean((V_ic_pred - V_ic_true)**2)

    # Calculating the boundary condition loss
    V_bc_pred = model(S_bc, v_bc, tau_bc)
    loss_bc = torch.mean((V_bc_pred - 0.0)**2)

    # Total Loss and Backpropagation
    loss = (LAMBDA_PDE * loss_pde) + (LAMBDA_IC * loss_ic) + (LAMBDA_BC * loss_bc)

    # Compute the gradients and update the neural network weights
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    
    # Record history for plotting
    history['epoch'].append(epoch)
    history['total'].append(loss.item())
    history['pde'].append(loss_pde.item())
    history['ic'].append(loss_ic.item())
    history['bc'].append(loss_bc.item())

    # Print rows of progress
    print_interval = max(1, epochs // PRINT_ROWS)
    if epoch % print_interval == 0:
        print(f"{epoch:6d} | {loss.item():12.6f} | {loss_pde.item():12.6f} | {loss_ic.item():12.6f} | {loss_bc.item():12.6f}")

# Stop that timer!
end_time = time.time()

# Calculate minutes and seconds
total_seconds = end_time - start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

# Calculate dataset metrics
n_interior = len(S_in)
n_ic = len(S_ic)
n_bc = len(S_bc)
total_points = n_interior + n_ic + n_bc

# Printout training complete with the final metrics
print("-" * 66)
print(f"Training Complete! Trained {epochs} epochs in {minutes} minutes and {seconds} seconds.")
print(f"Dataset Size: {total_points:,} total points ({n_interior:,} Interior, {n_ic:,} IC, {n_bc:,} BC)")
print(f"Final Total Loss: {loss.item():.6f}")
print("=" * 66)


### Pre-plotting


In [ ]:
# Creating a clean 50x50 grid for plotting (Holding v constant at initial v0)
S_plot = np.linspace(1e-5, S_max, 50)
tau_plot = np.linspace(0.01, T_max, 50)
S_grid, tau_grid = np.meshgrid(S_plot, tau_plot)

# Flatten the grids and convert to PyTorch tensors
S_flat = torch.tensor(S_grid.flatten(), dtype=torch.float32).view(-1, 1).to(device)
tau_flat = torch.tensor(tau_grid.flatten(), dtype=torch.float32).view(-1, 1).to(device)
v_flat = torch.full_like(S_flat, v0).to(device) # Constant variance

# Getting PINN predictions
with torch.no_grad():
    V_pred_flat = model(S_flat, v_flat, tau_flat).cpu().numpy()

# Reshape the flat predictions back into the 50x50 grid
pinn_surface = V_pred_flat.reshape(S_grid.shape)

# ===== NEW ANALYTICAL INTEGRATION =====
%run hs.ipynb

# Calculate the True Analytical Surface
print("Calculating Analytical Surface... this requires 5000 fast-Fourier integrations, may take a minute...")
analytical_surface = hs(
    S=S_grid, K=K, T=tau_grid, r=r, 
    v0=v0, kappa=kappa, theta=theta, xi=xi, rho=rho, 
    type="call"
)
print("Analytical Surface Generated!")


### Plotting the 3d surfaces


In [ ]:
fig = plt.figure(figsize=(16, 6))

# Plot 1: PINN Prediction (Fixing v=v0)
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(S_grid, tau_grid, pinn_surface, cmap=cm.viridis,
                         linewidth=0, antialiased=False, alpha=0.9)
ax1.set_xlabel('Stock Price (S)')
ax1.set_ylabel('Time to Maturity (tau)')
ax1.set_zlabel('Option Price (V)')
ax1.set_title(f'Heston Call (PINN Prediction, v={v0})')
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=0.5)

# Plot 2: Analytical Solution
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(S_grid, tau_grid, analytical_surface, cmap=cm.plasma,
                         linewidth=0, antialiased=False, alpha=0.9)
ax2.set_xlabel('Stock Price (S)')
ax2.set_ylabel('Time to Maturity (tau)')
ax2.set_zlabel('Option Price (V)')
ax2.set_title('Analytical Heston Solution (v=0.04)')
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=0.5)

plt.tight_layout()
plt.savefig(out_path_pinn, bbox_inches="tight")
plt.show()


### Plotting the loss curves


In [ ]:
df_hist = pd.DataFrame(history)

def plot_loss(loss_name, title, color, out_path):
    plt.figure(figsize=(10, 5))
    plt.plot(df_hist['epoch'], df_hist[loss_name], label=title, color=color)
    plt.yscale('log')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.title(f'Heston PINN - {title}')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()

# Export Total Loss
plot_loss('total', 'Total Loss', 'black', out_path_loss_total)


In [ ]:
# Export PDE Loss
plot_loss('pde', 'PDE Residual Loss', '#1f77b4', out_path_loss_pde)


In [ ]:
# Export IC Loss
plot_loss('ic', 'Initial Condition (IC) Loss', '#ff7f0e', out_path_loss_ic)

In [ ]:
# Export BC Loss
plot_loss('bc', 'Boundary Condition (BC) Loss', '#2ca02c', out_path_loss_bc)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted safely.")
